In [ ]:
%run init_notebook.py

import torch.nn as nn
from torchvision.datasets import FashionMNIST
from torch.utils.data import DataLoader
from torchvision.transforms import ToTensor
import matplotlib.pyplot as plt
import time

import torchaudio.transforms as T
import math   
from src.dataset import LatentNSynth 
import torch
from src.diffusion import *
from src.utils.models import adjust_shape, compute_magnitude_and_phase, compute_magnitude_and_phase_sin_cos
from src.utils.audio_utils import *


# Training steps:
0. Train a VAE on the training dataset to learn a compact latent representation of the audio data.

1. Precalculate latents from training and validation datasets using the trained VAE.

2. Train the diffusion model on the precalculated latents.

# Sampling from the latent diffusion model:

### I. Generate a latent representation using de diffusion model:
Load the trained diffusion model: $ D $

Generate a random noise image: $ z \sim \mathcal{N}(0, I) $

Iteratively denoise the image using the diffusion model:

$$ \hat{\epsilon} = D(z_t, t) $$

Update the image: $ z_{t-1} = \frac{1}{\sqrt{\alpha_t}} (z_t - \frac{1 - \alpha_t}{\sqrt{1 - \bar{\alpha}_t}} \hat{\epsilon}) + \sigma_t \cdot \epsilon_t $

Where $ \alpha_t $ and $ \bar{\alpha}_t $ are the noise schedule parameters, $ \sigma_t $ is the noise scale, and $ \epsilon_t \sim \mathcal{N}(0, I) $ is random noise.

### II. Decode the latent representation to audio:
Load the trained VAE decoder: $ V_{decoder} $

Decode the latent representation: $ x = V_{decoder}(z) $

Where $ x $ is the reconstructed audio signal.

In [ ]:
import json
from src.models import VAE
from src.diffusion_setups.setup_latent import setup_latent_model

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

paths = {
    "diffusion_model": r"C:\Users\Articuno\Desktop\TFG-MUSICAL\data\models\mod_stable_diff_music.pth",
    "diffusion_scheduler":  r"C:\Users\Articuno\Desktop\TFG-MUSICAL\data\models\sch_stable_diff_music.pth",
    "diffusion_config": r"C:\Users\Articuno\Desktop\TFG-MUSICAL\data\models\config_stable_diff_music.json",
    "vae_model": r"C:\Users\Articuno\Desktop\TFG-MUSICAL\data\models\vae_2D.pth"
}

# Audio processing parameters
sample_rate = 16000
n_fft = 1500
hop_length = 250
win_length = n_fft

istft_transform = T.transforms.InverseSpectrogram(
    n_fft=n_fft, win_length=win_length, hop_length=hop_length, onesided=False
).to(device)

def load_vae():
    model_path = paths["vae_model"]
    ''' Load the trained VAE model from the specified path.'''
    input_height = 1500
    input_width = 251
    input_size = (input_height, input_width)
    latent_dim = 200
    channels   = [2,16,32,64]
    model = VAE(input_size=input_size, latent_dim=latent_dim, channels=channels).to(device)
    model.load_state_dict(torch.load(model_path, map_location=device))
    model.eval()
    return model

def load_diffusion_model():
    config = {}
    with open(paths["diffusion_config"], "r") as f:
        config = json.load(f)
        
    model, scheduler, _ = setup_latent_model(
        timesteps=config["timesteps"],
        channels=config["channels"],
        norm_groups=config["norm_groups"],
        emb_dim=config["emb_dim"],   
    )
    model = torch.load(paths["diffusion_model"])
    scheduler = torch.load(paths["diffusion_scheduler"])
    model.to(device)
    scheduler.to(device)
    model.eval()
    scheduler.eval()
    return model, scheduler
    
    

In [ ]:
def sample_latent(D, D_scheduler, n=1):
    ''' Sample from the latent diffusion model.'''
    lh = 200
    lw = 200
    lc = 1
    
    z = torch.randn(n, lc, lh, lw).to(device)
    with torch.no_grad():
        for t in reversed(range(D_scheduler.num_train_timesteps)):
            t_batch = torch.full((n,), t, device=device, dtype=torch.long)
            e_pred = D(z, t_batch)
            beta_t = D_scheduler.betas[t]
            alpha_t = D_scheduler.alphas[t]
            alpha_bar_t = D_scheduler.alpha_bars[t]
            
            z = (1 / torch.sqrt(alpha_t)) * (z - ((1 - alpha_t) / torch.sqrt(1 - alpha_bar_t)) * e_pred)
            if t > 0:
                e_added = torch.randn_like(z)
                sigma_t = torch.sqrt(beta_t * (1 - D_scheduler.alpha_bars[t-1]) / (1 - D_scheduler.alpha_bars[t]))
                z += sigma_t * e_added
    return z

In [ ]:
# 1. Sample from the latent diffusion model
from matplotlib.style import use


D, D_scheduler = load_diffusion_model()
z = sample_latent(D, D_scheduler, n=1)
del D, D_scheduler # Free up memory

# 2. Decode the latent representation using the VAE
V = load_vae()
x = V.decode(z)
del V # Free up memory

# 3. Convert the spectrogram back to audio using the ISTFT
x = x.squeeze(0).cpu()  # Remove batch dimension and move to CPU
w = spectrogram_to_waveform(x, n_fft=n_fft, hop_length=hop_length, win_length=win_length, uselibrosa=False)

# PLOT EVERYTHING
plt.figure(figsize=(12, 6))
plt.subplot(1, 2, 1)
plt.title("Sampled Latent Representation (z)")
plt.imshow(z[0, 0].cpu(), aspect='auto', origin='lower')
plt.subplot(1, 2, 2)
plt.title("Decoded Spectrogram (x)")
plt.imshow(x[0, 0].cpu(), aspect='auto', origin='lower')

plot_waveform(w, sample_rate)
plot_spectrogram(x)
listen(w, sample_rate)